# 松尾研 Physical AI｜工程34：球体の誤進入を防ぐ物体認識実験
作成日時：2026/10/20 04:22 JST

**目的**：野球ボール・テニスボール・赤いゴムボールの3種類×各2条件＝計6試行で、
PyBullet実体（球体・質量・UVテクスチャ）を検査台に置く。
**実際のPyBullet上部カメラで撮影**し、3回の荷重測定が25g相当以上でも、
非りんごと識別した場合は3本指が下降しないことを確認する。

さらに**合成りんごA/B/C/D 4試行**を工程33と同条件で新たに実行して、りんごは通過し所定の受け台に置けることを確認する。

> これは**PyBulletによる仮想物体の試験**であり、実際の野球ボール・テニスボール・りんごではない。
> 途中で失敗しても判定を修正したことにせず、画像と個別結果をそのまま保存する。
> AIは運転前に1回だけ学習し、各試行では推論のみ実行する。


## 1｜工程34 ZIPをアップロードして展開

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # このセルの▶で新しくアップロード
candidates = [x for x in uploaded if x == 'PAI_stage34_negative_objects_20261020.zip']
assert len(candidates)==1, '工程34のZIPを一つ選んでください'
with zipfile.ZipFile(candidates[0]) as z:
    assert z.testzip() is None, 'ZIPが破損しています'
    z.extractall('/content')
root=Path('/content/PAI_stage34_negative_objects_20261020')
assert (root/'run_stage34_physics.py').is_file()
print('READY',root)


## 2｜ライブラリをインストール

In [ ]:
%cd /content/PAI_stage34_negative_objects_20261020
%pip install -r requirements.txt


## 3｜静的検査・画像AIのオフライン検査・事前学習

In [ ]:
!python -u test_stage34_offline.py
!python -u test_stage33_offline.py
!python -u run_stage34_physics.py --train-only


## 4｜非りんご球体6試行（荷重検知→実カメラ撮影→拒否→下降せず停止）
**注意：** りんごと誤認した試行は失敗として記録する。自動で閾値を変えない。

In [ ]:
!python -u run_stage34_physics.py --output stage34_negative_results


## 5｜りんご4等級の物理搬送を再確認（工程33の制御をそのまま実行）

In [ ]:
!python -u run_stage33_physics.py --mode normal --output stage34_positive_regression


## 6｜結果の独立監査・ZIP保存・ダウンロード
失敗した場合でも個別の試行と画像を含めてZIPを保存する。

In [ ]:
from pathlib import Path
import subprocess,zipfile,json
from google.colab import files
root=Path('/content/PAI_stage34_negative_objects_20261020')
neg=root/'stage34_negative_results'
pos=root/'stage34_positive_regression'
if neg.exists():
    cmd=['python',str(root/'audit_stage34_results.py'),str(neg),str(pos)]
    result=subprocess.run(cmd,capture_output=True,text=True)
    print('AUDIT_EXIT',result.returncode)
    print(result.stdout[-6000:]);print(result.stderr[-2000:])
else:print('NO_NEGATIVE_RESULTS: preceding simulation may have failed before output')
if pos.exists():
    subprocess.run(['python',str(root/'audit_stage33_results.py'),str(pos)],check=False)
archive=Path('/content/stage34_pybullet_object_gate_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for rel in ('OFFLINE_STATIC_STAGE34.json','OFFLINE_STAGE33_RESULTS.json'):
        p=root/rel
        if p.is_file():z.write(p,arcname=rel)
    m=root/'models'/'training_manifest.json'
    if m.is_file():z.write(m,arcname='models/training_manifest.json')
    for out in (neg,pos):
        if not out.is_dir():continue
        for p in sorted(out.rglob('*')):
            if not p.is_file():continue
            if 'engine' in p.relative_to(out).parts:continue  # 原コード96ファイルは同梱済み
            if p.suffix.lower() in ('.json','.csv','.png','.txt'):
                z.write(p,arcname=str(p.relative_to(root)))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    print('ZIP_FILE_COUNT',len(z.namelist()))
print('RESULT',archive,archive.stat().st_size,'bytes')
files.download(str(archive))
